Woah sick

In [1]:
!pip install pyspark pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
print(os.getcwd())

/expanse/lustre/projects/uci157/tragus


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = (SparkSession.builder.appName("MusicBrainz").config("spark.driver.memory", "2g").config("spark.executor.memory", "18g").config('spark.executor.instances', 7).getOrCreate())

Okay so here's the deal - it would be really great to use SQL-style manipulation during the processing stage, but we can't launch a SQL server from this environment. sqlite and DuckDB are apparently local solutions that work around this bottleneck (they don't use a server and can be run within a Jupyter environment) and are worth exploring here.

As a proof of concept my goal tonight is just to get some tables loaded with Spark

In [4]:
MBDUMP = "musicbrainz_project/raw_data/mbdump"

In [5]:
def peek(df, n=20):
    return display(df.limit(n).toPandas())

ARTISTS

In [6]:
artist_schema = StructType([
    StructField("id",               IntegerType(),   True),
    StructField("gid",              StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("sort_name",        StringType(),    True),
    StructField("begin_date_year",  IntegerType(),   True),
    StructField("begin_date_month", IntegerType(),   True),
    StructField("begin_date_day",   IntegerType(),   True),
    StructField("end_date_year",    IntegerType(),   True),
    StructField("end_date_month",   IntegerType(),   True),
    StructField("end_date_day",     IntegerType(),   True),
    StructField("type",             IntegerType(),   True),
    StructField("area",             IntegerType(),   True),
    StructField("gender",           IntegerType(),   True),
    StructField("comment",          StringType(),    True),
    StructField("edits_pending",    IntegerType(),   True),
    StructField("last_updated",     StringType(),    True),
    StructField("ended",            StringType(),    True),
    StructField("begin_area",       IntegerType(),   True),
    StructField("end_area",         IntegerType(),   True),
])

artist_df = (
    spark.read
    .option("sep", "\t")
    .option("nullValue", r"\N")
    .option("header", "false")
    .option("quote", "")
    .option("escape", "")
    .schema(artist_schema)
    .csv(f"{MBDUMP}/artist")
)

print("=== SCHEMA ===")
artist_df.printSchema()

# Peek in
peek(artist_df)

# Define Spark RDD
artist_rdd = artist_df.rdd
print("Total artists:")
artist_rdd.count()

=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sort_name: string (nullable = true)
 |-- begin_date_year: integer (nullable = true)
 |-- begin_date_month: integer (nullable = true)
 |-- begin_date_day: integer (nullable = true)
 |-- end_date_year: integer (nullable = true)
 |-- end_date_month: integer (nullable = true)
 |-- end_date_day: integer (nullable = true)
 |-- type: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- ended: string (nullable = true)
 |-- begin_area: integer (nullable = true)
 |-- end_area: integer (nullable = true)



,id,gid,name,sort_name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,type,area,gender,comment,edits_pending,last_updated,ended,begin_area,end_area
0,2252039,fadeb38c-833f-40bc-9d8c-a6383b38b1be,Доктор Сатана,Доктор Сатана,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2021-11-23 07:08:52.479537+00,f,NaN,NaN
1,371203,49add228-eac5-4de8-836c-d75cde7369c3,Pete Moutso,"Moutso, Pete",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,None,0,None,f,NaN,NaN
2,3087346,dfdce491-133d-4e9f-9e48-795587e181b0,UNlT,UNlT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2025-09-14 23:53:28.004798+00,f,NaN,NaN
3,2851271,165a49a0-2b3b-4078-a3c1-905afdc07c0a,Babyglock,Babyglock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2024-10-19 03:33:55.474151+00,f,NaN,NaN
4,145773,7b4a548e-a01a-49b7-82e7-b49efeb9732c,Aric Leavitt,"Leavitt, Aric",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
5,1076328,60aca66f-e91a-4cb5-9308-b6e293cd833e,Fonograff,Fonograff,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2014-01-10 16:25:20.992213+00,f,NaN,NaN
6,1172876,3e1bd546-d2a7-49cb-b38d-d70904a1d719,Al Street,"Street, Al",NaN,NaN,NaN,NaN,NaN,NaN,1.0,222.0,1.0,None,0,2014-11-23 14:07:19.782509+00,f,NaN,NaN
7,220155,df120895-f6c6-4a66-b9cf-73350f0beb61,Love .45,Love .45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
8,618464,c14f8d3f-ee81-416f-800f-8eff7e77a2e1,Sintellect,Sintellect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2009-05-23 09:41:53.269195+00,f,NaN,NaN
9,285714,b68a3969-319a-462f-942b-cd35581414fc,Evie Tamala,Evie Tamala,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN


Total artists:


2853154

GENRE

In [7]:
genre_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("gid",            StringType(),  True),
    StructField("name",           StringType(),  True),
    StructField("comment",        StringType(),  True),
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
])

genre_df = (
    spark.read
    .option("sep", "\t")
    .option("nullValue", r"\N")
    .option("header", "false")
    .option("quote", "")
    .option("escape", "")
    .schema(genre_schema)
    .csv(f"{MBDUMP}/genre")
)

print("=== SCHEMA ===")
genre_df.printSchema()

# Peek in
peek(genre_df)

# Define Spark RDD
genre_rdd = genre_df.rdd
print("Total genres:")
genre_rdd.count()

=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: string (nullable = true)



,id,gid,name,comment,edits_pending,last_updated
0,1,54c01942-22fd-4184-9877-1db0089b18f1,acid house,None,0,2019-05-13 17:46:28.122726+00
1,2,7dc2b20f-3953-4874-b9bf-41b8ba06d20c,acid jazz,None,0,2019-05-13 17:46:28.122726+00
2,3,ba64013e-27bb-4f14-a530-8d25b296e0da,acid techno,None,0,2019-05-13 17:46:28.122726+00
3,4,37f85b9c-c3fc-4b5a-8545-51aeb78c8786,acoustic blues,None,0,2019-05-13 17:46:28.122726+00
4,5,00055e8b-b951-46e2-af1e-58b5624e7952,acoustic rock,None,0,2019-05-13 17:46:28.122726+00
5,1754,a7e0229c-6e53-45f1-a6f2-a791e78b159e,afro-zouk,None,0,2022-12-21 11:47:31.756858+00
6,7,5f9cba3d-1a9f-46cd-8c49-7ed78d1f3354,alternative country,None,0,2019-05-13 17:46:28.122726+00
7,8,8301f73c-9166-4108-bfeb-4fd22dc19083,alternative dance,None,0,2019-05-13 17:46:28.122726+00
8,9,0b48a36c-630f-4ee7-8cf3-480e3dd8be65,alternative folk,None,0,2019-05-13 17:46:28.122726+00
9,10,924943cd-73c8-45c0-96eb-74f2a15e5d6e,alternative hip hop,None,0,2019-05-13 17:46:28.122726+00


Total genres:


2133

LABEL

In [8]:
label_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("gid",               StringType(),  True),
    StructField("name",              StringType(),  True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("label_code",        IntegerType(), True),
    StructField("type",              IntegerType(), True),
    StructField("area",              IntegerType(), True),
    StructField("comment",           StringType(),  True),
    StructField("edits_pending",     IntegerType(), True),
    StructField("last_updated",      StringType(),  True),
    StructField("ended",             StringType(),  True),
])

label_df = (
    spark.read
    .option("sep", "\t")
    .option("nullValue", r"\N")
    .option("header", "false")
    .option("quote", "")
    .option("escape", "")
    .schema(label_schema)
    .csv(f"{MBDUMP}/label")
)

print("=== SCHEMA ===")
label_df.printSchema()

# Peek in
peek(label_df)

# Define Spark RDD
label_rdd = label_df.rdd
print("Total labels:")
label_rdd.count()

=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- begin_date_year: integer (nullable = true)
 |-- begin_date_month: integer (nullable = true)
 |-- begin_date_day: integer (nullable = true)
 |-- end_date_year: integer (nullable = true)
 |-- end_date_month: integer (nullable = true)
 |-- end_date_day: integer (nullable = true)
 |-- label_code: integer (nullable = true)
 |-- type: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- ended: string (nullable = true)



,id,gid,name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,label_code,type,area,comment,edits_pending,last_updated,ended
0,1,f43e252d-9ebf-4e8e-bba8-36d080756cc1,Deleted Label,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f
1,2,39c4dc0c-badb-4ac3-b810-e4f374dff6d9,Certificate 18,NaN,NaN,NaN,NaN,NaN,NaN,2592.0,4.0,221.0,None,0,None,f
2,103730,6f70a5cb-99a7-4a42-9208-412446d4aa0f,Flo Master Inc.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,222.0,None,0,2015-05-18 20:41:54.551166+00,f
3,29683,ccbbf728-15b8-43ee-91b6-b06967ed7f83,Xunk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,73.0,None,0,None,f
4,195899,953f5437-c702-4aaf-b7c6-d4055fe9b21b,Brother Studio Productions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,30646.0,None,0,2020-06-05 15:54:50.330605+00,f
5,16711,1e742d6b-47ea-4e53-9c64-5813b2510463,Isma'a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,73.0,None,0,None,f
6,6,5e39ea9e-3d0f-4880-8ce2-fbe561241538,Svek,1996.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,202.0,None,0,None,f
7,7768,5fb67896-f6d3-49f5-a8de-65ff17ad2dbe,Rock n' Roll Radio,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,None,0,None,f
8,294616,45b14197-41ac-4ef8-914b-31a7362b1371,senpaiこみゅ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,107.0,doujin circle,0,2024-05-13 20:27:01.073865+00,f
9,146127,5b22cc47-5384-433b-82e3-5b5cab11a8c9,Eurobeat Union,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2017-12-10 20:06:56.906165+00,f


Total labels:


334811